# WhatsApp Auto-Reply Bot — CRM Integration Module
**Week 2 individual task — Group 53, SafeX Solutions ML Internship**

This notebook demonstrates the CRM Integration module in isolation:
- Takes sample "captured leads" (as they'd arrive from a teammate's WhatsApp
  message-capture module)
- Normalizes phone numbers and **de-duplicates** leads
- **Tags** each message with an intent (Pricing / Demo Request / Support / Purchase Intent / General)
- Creates or updates records in the CRM

For this demo we use `MockAirtableClient` (in-memory, no credentials needed)
so it runs anywhere. Swapping in the real `AirtableClient` (see
`app/airtable_client.py`) requires no changes to the pipeline logic itself —
just set the `AIRTABLE_API_KEY` / `AIRTABLE_BASE_ID` environment variables.

In [1]:
import json
import sys
sys.path.insert(0, "..")  # if running notebook from a subfolder, adjust as needed

from app.schema import IncomingLead
from app.crm import process_lead
from app.airtable_client import MockAirtableClient

with open("sample_leads.json") as f:
    sample_leads = json.load(f)

sample_leads

[{'name': 'Ali Raza',
  'phone': '0300-1234567',
  'message': 'Hi, what is the price of your service?',
  'source': 'WhatsApp'},
 {'name': None,
  'phone': '+92 321 9876543',
  'message': 'I want a demo of your product please',
  'source': 'WhatsApp'},
 {'name': 'Ali Raza',
  'phone': '03001234567',
  'message': 'Following up, is there a discount available?',
  'source': 'WhatsApp'},
 {'name': 'Sara Khan',
  'phone': '0333-4445556',
  'message': 'My last order had an issue, it is not working',
  'source': 'WhatsApp'},
 {'name': 'Ali Raza',
  'phone': '92 300 1234567',
  'message': "Ok I'd like to buy the premium plan",
  'source': 'WhatsApp'}]

## Run the pipeline on sample leads

In [2]:
client = MockAirtableClient()
results = []

for raw in sample_leads:
    lead = IncomingLead(**raw)
    result = process_lead(lead, client)
    results.append(result)
    print(f"{result['action']:20s} | phone={result['record']['fields']['Phone']:15s} "
          f"| intent={result['record']['fields']['Intent Tag']:16s} "
          f"| status={result['record']['fields']['Status']:10s} "
          f"| msg_count={result['record']['fields']['Message Count']}")

created              | phone=923001234567    | intent=Pricing          | status=New        | msg_count=1
created              | phone=923219876543    | intent=Demo Request     | status=New        | msg_count=1
updated_duplicate    | phone=923001234567    | intent=Pricing          | status=Contacted  | msg_count=2
created              | phone=923334445556    | intent=Support          | status=New        | msg_count=1
updated_duplicate    | phone=923001234567    | intent=Purchase Intent  | status=Contacted  | msg_count=3


## Inspect the resulting CRM state

Notice: the 5 sample leads collapse to **3 CRM records** because 3 of them
belonged to Ali Raza across different phone-number formats
(`0300-1234567`, `03001234567`, `92 300 1234567` all normalize to the same
number) — this is the de-duplication in action. His record's status moved
from `New` -> `Contacted`, and `Message Count` incremented each time.

In [3]:
all_records = client.get_all_leads()
for r in all_records:
    print(r["fields"])

{'Name': 'Ali Raza', 'Phone': '923001234567', 'Last Message': "Ok I'd like to buy the premium plan", 'Intent Tag': 'Purchase Intent', 'Status': 'Contacted', 'Source': 'WhatsApp', 'First Contact': '2026-07-16T19:13:38.575381+00:00', 'Last Contact': '2026-07-16T19:13:38.604651+00:00', 'Message Count': 3}
{'Name': 'Unknown', 'Phone': '923219876543', 'Last Message': 'I want a demo of your product please', 'Intent Tag': 'Demo Request', 'Status': 'New', 'Source': 'WhatsApp', 'First Contact': '2026-07-16T19:13:38.582777+00:00', 'Last Contact': '2026-07-16T19:13:38.582777+00:00', 'Message Count': 1}
{'Name': 'Sara Khan', 'Phone': '923334445556', 'Last Message': 'My last order had an issue, it is not working', 'Intent Tag': 'Support', 'Status': 'New', 'Source': 'WhatsApp', 'First Contact': '2026-07-16T19:13:38.597704+00:00', 'Last Contact': '2026-07-16T19:13:38.597704+00:00', 'Message Count': 1}


## Sample input / output summary

| # | Input phone (raw) | Normalized | Action | Intent | Status |
|---|---|---|---|---|---|
| 1 | 0300-1234567 | 923001234567 | created | Pricing | New |
| 2 | +92 321 9876543 | 923219876543 | created | Demo Request | New |
| 3 | 03001234567 | 923001234567 | updated_duplicate | Pricing | Contacted |
| 4 | 0333-4445556 | 923334445556 | created | Support | New |
| 5 | 92 300 1234567 | 923001234567 | updated_duplicate | Purchase Intent | Contacted |

## How to run this for real
1. Create an Airtable base with a `Leads` table (see `README.md` for the exact field list).
2. Set env vars: `AIRTABLE_API_KEY`, `AIRTABLE_BASE_ID`, `AIRTABLE_TABLE_NAME`.
3. Replace `MockAirtableClient()` above with `AirtableClient()` — no other code changes needed.
4. Or run the FastAPI service (`uvicorn app.main:app --reload --port 8001`) so teammates' modules can POST leads directly to `/leads/webhook`.